# Documentation

This code calculates the normalized gross moist stability (nGMS) following Inoue & Back (2015).

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# Aquaplanet analysis config file
import config
import coords

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

logger.info("Loading imports...")

# File management
import sys

# Data anaylsis
import numpy as np
import xarray as xr
import xeofs as xe
xr.set_options(keep_attrs=True)
from scipy.stats import t

# Plotting
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import ticker as mticker
from matplotlib.gridspec import GridSpec
plt.rcParams['mathtext.fontset'] = 'dejavusans'
import string

# Auxiliary functions
from load_aquaplanet_data import load_multi_experiment_processed_data
from auxiliary_functions.xarray_utils import standardize_data#, add_cyclic_xarray
from auxiliary_functions.plotting_utils import bmh_colors, tick_labeller, set_plot_mode, get_figsize
from processing_functions import time_filter_data, mjo_filter_data

logger.info("Imports loaded")

# Calculate nGMS constituents

In [ ]:
# save_column_integrated_data = True

# # Initialize arrays for budget variables
# column_horizontal_MSE_advection = {}
# column_vertical_MSE_advection = {}
# column_vertical_DSE_advection = {}

# n_calculations = 3

# variables_loaded = {}
# for exp_index, experiment in enumerate(experiments_list):

#     print(f"{'-'*config.SEP_WIDTH}")
#     print(f"{f'Loading {experiment} data...':<{config.SEP_WIDTH}}")
#     print(f"{'-'*config.SEP_WIDTH}")
#     variable_data_files = sorted(glob.glob(
#         rf"{data_directory}/{experiment}/daily_model-level_data/*.nc"
#     ))
#     for index, file in enumerate(variable_data_files):
#         variable_data = xr.open_dataarray(file)
#         print(f"{f'({index+1}/{len(variable_data_files)}) {variable_data.name}...':<{config.SEP_WIDTH-1}}", end="")
#         variables_loaded[variable_data.name] = variable_data.sel(time=slice(START_TIME, END_TIME))
#         print(rf"{'✔':>1}")
#     print(f"{'-'*config.SEP_WIDTH}")

#     recalculate_budgets = True

#     non_timed_data =  xr.open_dataset(
#         rf"/glade/campaign/univ/uwas0114/SST_AQP3_Qobs_27_-4K_3h_10y/atm/hist/SST_AQP3_Qobs_27_-4K_3h_20y_new2.cam.h1.0001-02-16-43200.nc"
#     )

#     if recalculate_budgets:
#         print(f"{'Pressure Array...':<{config.SEP_WIDTH-1}}", end="")
#         lower_level_pressure = 1100.*100.
#         # upper_level_pressure = 100.*100.
#         surface_pressure = variables_loaded['PS']
#         pressure_array = non_timed_data['hyam']*non_timed_data['P0'] + non_timed_data['hybm']*surface_pressure
#         pressure_array = pressure_array.transpose("time", "lev", "lat", "lon")
#         print(rf"{'✔':>1}")

#         time_mean_temperature = variables_loaded['T'].mean(dim=['time'])
#         temperature_minimum_index = time_mean_temperature.argmin('lev')
#         temperature_minimum_pressure_level = pressure_array.mean(dim='time').transpose("lev", ...)[temperature_minimum_index].mean(dim=['lat', 'lon'])
#         upper_level_pressure = temperature_minimum_pressure_level.values

#         print(f"{'Moist Static Energy...':<{config.SEP_WIDTH-1}}", end="")
#         GRAVITY = 9.8                        #  m/s^2
#         SPECIFIC_HEAT = 1005.7               #  J/Kg*K ; specific heat at constant pressure for dry air
#         HEAT_OF_VAPORIZATION = 2.501e6       #  [J/kg]=[m2/s2]  Latent Heat of Vaporization at 0
#         HEAT_OF_FUSION = 3.337e5             # [J/kg]=[m2/s2]  Latent Heat of Sublimation at 0
#         dry_static_energy = (
#             + SPECIFIC_HEAT*variables_loaded['T']
#             + GRAVITY*variables_loaded['Z3']
#         )
#         dry_static_energy.name = 'Dry Static Energy'
#         dry_static_energy.attrs['units'] = r"J kg$^{-2}$"

#         moist_static_energy = (
#             HEAT_OF_VAPORIZATION*variables_loaded['Q']
#             - HEAT_OF_FUSION*variables_loaded['CLDICE']
#             + SPECIFIC_HEAT*variables_loaded['T']
#             + GRAVITY*variables_loaded['Z3']
#         )
#         moist_static_energy.name = 'Moist Static Energy'
#         moist_static_energy.attrs['units'] = r"J kg$^{-2}$"
#         print(rf"{'✔':>1}")


#         print(f"{'Zonal Advection...':<{config.SEP_WIDTH-1}}", end="")
#         zonal_MSE_gradient = (
#             (180/np.pi)
#             * moist_static_energy.differentiate('lon')
#             / (EARTH_RADIUS*np.cos(np.deg2rad(moist_static_energy.lat)))
#         )
#         zonal_advection = variables_loaded['U']*zonal_MSE_gradient
#         zonal_advection.name = 'Zonal Advection'
#         zonal_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"
#         print(rf"{'✔':>1}")

#         print(f"{'Meridional Advection...':<{config.SEP_WIDTH-1}}", end="")
#         meridional_MSE_gradient = (
#             (180/np.pi)
#             * moist_static_energy.differentiate('lat')
#             / EARTH_RADIUS
#         )

#         meridional_advection = variables_loaded['V'] * meridional_MSE_gradient
#         meridional_advection.name = 'Meridional Advection'
#         meridional_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"
#         print(rf"{'✔':>1}")

#         horizontal_MSE_advection = zonal_advection + meridional_advection
#         horizontal_MSE_advection.name = 'Horizontal Advection'
#         horizontal_MSE_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"

#         print(f"{'Vertical MSE Advection...':<{config.SEP_WIDTH-1}}", end="")
#         # vertical_MSE_gradient = (1/100)*moist_static_energy.differentiate('lev')
#         vertical_MSE_gradient = xr.zeros_like(moist_static_energy)
#         vertical_MSE_gradient[:, 0] = (
#             (moist_static_energy[:, 1].values - moist_static_energy[:, 0].values)
#             / (pressure_array[:, 1].values - pressure_array[:, 0].values)
#         )
#         vertical_MSE_gradient[:, 1:-1] = (
#             (moist_static_energy[:, 2:].values - moist_static_energy[:, :-2].values)
#             / (pressure_array[:, 2:].values - pressure_array[:, :-2].values)
#         )
#         vertical_MSE_gradient[:, -1] = (
#             (moist_static_energy[:, -1].values - moist_static_energy[:, -2].values)
#             / (pressure_array[:, -1].values - pressure_array[:, -2].values)
#         )

#         vertical_MSE_advection = variables_loaded['OMEGA'] * vertical_MSE_gradient
#         vertical_MSE_advection.name = 'Vertical MSE Advection'
#         vertical_MSE_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"
#         print(rf"{'✔':>1}")

#         print(f"{'Vertical DSE Advection...':<{config.SEP_WIDTH-1}}", end="")
#         # vertical_MSE_gradient = (1/100)*moist_static_energy.differentiate('lev')
#         vertical_DSE_gradient = xr.zeros_like(dry_static_energy)
#         vertical_DSE_gradient[:, 0] = (
#             (dry_static_energy[:, 1].values - dry_static_energy[:, 0].values)
#             / (pressure_array[:, 1].values - pressure_array[:, 0].values)
#         )
#         vertical_DSE_gradient[:, 1:-1] = (
#             (dry_static_energy[:, 2:].values - dry_static_energy[:, :-2].values)
#             / (pressure_array[:, 2:].values - pressure_array[:, :-2].values)
#         )
#         vertical_DSE_gradient[:, -1] = (
#             (dry_static_energy[:, -1].values - dry_static_energy[:, -2].values)
#             / (pressure_array[:, -1].values - pressure_array[:, -2].values)
#         )

#         vertical_DSE_advection = variables_loaded['OMEGA'] * vertical_DSE_gradient
#         vertical_DSE_advection.name = 'Vertical DSE Advection'
#         vertical_DSE_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"
#         print(rf"{'✔':>1}")

#         print(f"{'Column integrating...':<{config.SEP_WIDTH-1}}", end="")
#         # Horizontal MSE Advection
#         column_horizontal_MSE_advection[experiment] = beta_column_integrate(
#             pressure_array,
#             horizontal_MSE_advection,
#             surface_pressure,
#             lower_level_pressure,
#             upper_level_pressure
#         )
#         column_horizontal_MSE_advection[experiment].name = 'Horizontal MSE Advection'
#         column_horizontal_MSE_advection[experiment].attrs = {}
#         column_horizontal_MSE_advection[experiment].attrs['long_name'] = "Horizontal of Moist Static Energy"
#         column_horizontal_MSE_advection[experiment].attrs['math_name'] = r"$\langle$$\vec{v} \nabla$h$\rangle$"
#         column_horizontal_MSE_advection[experiment].attrs['units'] = r"W m$^{-2}$"

#         # Vertical MSE Advection
#         column_vertical_MSE_advection[experiment] = beta_column_integrate(
#             pressure_array,
#             vertical_MSE_advection,
#             surface_pressure,
#             lower_level_pressure,
#             upper_level_pressure
#         )
#         column_vertical_MSE_advection[experiment].name = 'Vertical MSE Advection'
#         column_vertical_MSE_advection[experiment].attrs = {}
#         column_vertical_MSE_advection[experiment].attrs['long_name'] = "Vertical Advection of Moist Static Energy"
#         column_vertical_MSE_advection[experiment].attrs['math_name'] = r"$-\langle$$ω \partial_{p}$h$\rangle$"
#         column_vertical_MSE_advection[experiment].attrs['units'] = r"W m$^{-2}$"

#         # Vertical MSE Advection
#         column_vertical_DSE_advection[experiment] = beta_column_integrate(
#             pressure_array,
#             vertical_DSE_advection,
#             surface_pressure,
#             lower_level_pressure,
#             upper_level_pressure
#         )
#         column_vertical_DSE_advection[experiment].name = 'Vertical DSE Advection'
#         column_vertical_DSE_advection[experiment].attrs = {}
#         column_vertical_DSE_advection[experiment].attrs['long_name'] = "Vertical Advection of Dry Static Energy"
#         column_vertical_DSE_advection[experiment].attrs['math_name'] = r"$-\langle$$ω \partial_{p}$d$\rangle$"
#         column_vertical_DSE_advection[experiment].attrs['units'] = r"W m$^{-2}$"
#         print(rf"{'✔':>1}")

# GMS_variables = {
#     'Horizontal MSE Advection': column_horizontal_MSE_advection,
#     'Vertical MSE Advection': column_vertical_MSE_advection,
#     'Vertical DSE Advection': column_vertical_DSE_advection,
# }

# print("Concatenating along experiment axis...")
# print(f"{'-'*config.SEP_WIDTH}")
# multi_experiment_GMS_variables = {}
# for index, GMS_variable in enumerate(GMS_variables):
#     print(f"{f'({index+1}/{len(GMS_variables)}) {GMS_variable}...':<{config.SEP_WIDTH-1}}", end="")
#     multi_experiment_GMS_variables[GMS_variable] = xr.concat(
#         [GMS_variables[GMS_variable][experiment] for experiment in experiments_list],
#         dim=experiments_list
#     )
#     multi_experiment_GMS_variables[GMS_variable] = multi_experiment_GMS_variables[GMS_variable].rename(
#         {"concat_dim": "experiment"}
#     )
#     print(rf"{'✔':>1}")

# if save_column_integrated_data:
#     print(f"{'Saving budget terms':^{config.SEP_WIDTH}}")
#     print(f"{'='*config.SEP_WIDTH}")

#     for index, (variable_name, variable_data) in enumerate(multi_experiment_GMS_variables.items()):
#         print(f"{f'({index+1}/{len(multi_experiment_GMS_variables)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

#         filename = f"{data_directory}/MSE_budget_terms/daily_model-level_data/multi_experiment_{variable_name.lower().replace(' ', '_')}.nc"
#         if os.path.exists(filename):
#             # Prompt user for confirmation
#             # response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
#             response = 'y'
#             if response == 'y':
#                 os.remove(filename)  # Delete the existing file
#                 variable_data.to_netcdf(filename)  # Save the new file
#                 print(rf"{'✔ (overwritten)':>1}")
#             else:
#                 print(rf"{'✘ (skipped)':>1}")
#         else:
#             variable_data.to_netcdf(filename)  # Save the new file
#             print(rf"{'✔':>1}")
# else:
#     print(f"{'Not saving budget terms':<{config.SEP_WIDTH}}")

# print(f"{'='*config.SEP_WIDTH}")
# print("Finished")

# Load data

In [ ]:
variables_to_load = [
    "Precipitation",
    "Outgoing Longwave Radiation",
    # "Zonal Wind",
    # "Meridional Wind",
    # "Vertical Wind",
    # "Temperature",
    # "Moisture",
    # "Relative Humidity",
    # "Geopotential Height",
    # "Longwave Heating Rate",
    # "Shortwave Heating Rate",
    # "Latent Heat Flux",
    # "Sensible Heat Flux",
    # "Surface Pressure",
    # "Moist Static Energy",
    # "Dry Static Energy",
    # "Column Moist Static Energy",
    # "Column Water Vapor",
    # "Column Temperature",
    # "Column Longwave Heating",
    # "Column Shortwave Heating",
    # "Potential Temperature",
    # "Saturation Specific Humidity"
    # "Column Relative Humidity",
    # "Diabatic Heating",
    # "Chikira alpha"
    # "Surface Longwave Flux",
    # "Surface Shortwave Flux",
    # "TOA Longwave Flux",
    # "TOA Shortwave Flux",
    # "Net Longwave Flux",
    # "Net Shortwave Flux",
    # "Cloud Ice Water Content",
    # "Cloud Fraction"
]

## Subset data

In [ ]:
if not 'multi_experiment_variables_subset' in locals():
    multi_experiment_variables_subset = load_multi_experiment_processed_data(
        variables_to_load,
        'subset'
    )
else:
    multi_experiment_variables_subset = load_multi_experiment_processed_data(
        variables_to_load,
        'subset',
        multi_experiment_variables_subset,
        False
    )

## Filtered data

In [ ]:
if not 'multi_experiment_variables_filtered' in locals():
    multi_experiment_variables_filtered = load_multi_experiment_processed_data(
        # variables_to_load,
        ['Precipitation', 'Outgoing Longwave Radiation',],#'Column Longwave Heating', 'Column Shortwave Heating'],
        'filtered'
    )
else:
    multi_experiment_variables_filtered = load_multi_experiment_processed_data(
        # variables_to_load,
        ['Precipitation', 'Outgoing Longwave Radiation',],#'Column Longwave Heating', 'Column Shortwave Heating'],
        'filtered',
        multi_experiment_variables_filtered,
        False
    )

## MJO-filtered

In [ ]:
if not 'multi_experiment_variables_mjo_filtered' in locals():
    multi_experiment_variables_mjo_filtered = load_multi_experiment_processed_data(
        # variables_to_load,
        ['Precipitation', 'Outgoing Longwave Radiation',],# 'Column Longwave Heating', 'Column Shortwave Heating'],
        'mjo_filtered'
    )
else:
    multi_experiment_variables_mjo_filtered = load_multi_experiment_processed_data(
        # variables_to_load,
        ['Precipitation', 'Outgoing Longwave Radiation',],# 'Column Longwave Heating', 'Column Shortwave Heating'],
        'mjo_filtered',
        multi_experiment_variables_mjo_filtered,
        False
    )

In [ ]:
multi_experiment_GMS_variables = {}
for variable_name in [
    'Horizontal MSE Advection',
    'Vertical MSE Advection',
    'Vertical DSE Advection',
]:
    multi_experiment_GMS_variables[variable_name] = xr.load_dataset(
        f"{config.DATA_DIRECTORY}/MSE_budget_terms/daily_model-level_data/multi_experiment_{variable_name.lower().replace(' ', '_')}.nc"
    )[variable_name].drop_sel(time=coords.missing_days, errors='ignore')
    multi_experiment_GMS_variables[variable_name].name = variable_name

multi_experiment_GMS_variables['Total MSE Advection'] = (
    multi_experiment_GMS_variables['Vertical MSE Advection']
    + multi_experiment_GMS_variables['Horizontal MSE Advection']
)

## Intraseasonally Filter GMS Variables

In [ ]:
multi_experiment_intraseasonally_filtered_GMS_variables = {}

print(f"{f'Filtering variables...':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")
for index, (variable_name, variable_data) in enumerate(multi_experiment_GMS_variables.items()):
    print(f"{f'({index+1}/{len(multi_experiment_GMS_variables)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")
    multi_experiment_intraseasonally_filtered_GMS_variables[variable_name] = {}
    for experiment in coords.experiments.values:
        first_half = time_filter_data(variable_data.sel(
            experiment=experiment,
            time=coords.first_half_subset_bounds
        ), slice(100, 20))
        second_half = time_filter_data(variable_data.sel(
            experiment=experiment,
            time=coords.second_half_subset_bounds
        ), slice(100, 20))

        multi_experiment_intraseasonally_filtered_GMS_variables[variable_name][experiment] = xr.concat(
            [first_half, second_half], dim='time'
        ).drop_sel(time=coords.missing_days, errors='ignore')

    multi_experiment_intraseasonally_filtered_GMS_variables[variable_name] = xr.concat(
        [multi_experiment_intraseasonally_filtered_GMS_variables[variable_name][experiment] for experiment in coords.experiments.values],
        dim=coords.experiments
    )
    print(rf"{'✔':>1}")
print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
correlation = xr.dot(
    standardize_data(multi_experiment_intraseasonally_filtered_GMS_variables['Vertical MSE Advection'], dim='time', unit_variance=True),
    standardize_data(multi_experiment_intraseasonally_filtered_GMS_variables['Vertical DSE Advection'], dim='time', unit_variance=True),
    dim='time'
) / len(multi_experiment_intraseasonally_filtered_GMS_variables['Vertical DSE Advection'].time)

## MJO Filter GMS Variables

In [ ]:
print(f"{'MJO filter variables':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

variables_mjo_filtered = {}
multi_experiment_mjo_filtered_GMS_variables = {}

for index, (variable_name, variable_data) in enumerate(multi_experiment_intraseasonally_filtered_GMS_variables.items()):
    multi_experiment_mjo_filtered_GMS_variables[variable_name] = {}
    print(f"{f'({index+1}/{len(multi_experiment_intraseasonally_filtered_GMS_variables)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")
    for experiment in coords.experiments.values:

        first_half = mjo_filter_data(
            variable_data.sel(experiment=experiment, time=coords.first_half_subset_bounds),
            wavenumber_bounds=slice(1, 3)
        ).real

        second_half = mjo_filter_data(
            variable_data.sel(experiment=experiment, time=coords.second_half_subset_bounds),
            wavenumber_bounds=slice(1, 3)
        ).real

        multi_experiment_mjo_filtered_GMS_variables[variable_name][experiment] = xr.concat(
            (
                first_half,
                second_half,
            ),
            dim='time'
        )

    multi_experiment_mjo_filtered_GMS_variables[variable_name] = xr.concat(
        [multi_experiment_mjo_filtered_GMS_variables[variable_name][experiment] for experiment in coords.experiments.values],
        dim=coords.experiments
    )

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

## Calculate mean nGMS quantities

In [ ]:
time_mean_nGMS = (
    multi_experiment_GMS_variables['Horizontal MSE Advection']
    + multi_experiment_GMS_variables['Vertical MSE Advection']
).mean(dim=['time']) / (multi_experiment_GMS_variables['Vertical DSE Advection']).mean(dim='time')

spatial_mean_nGMS = xr.concat(
    [(
            multi_experiment_GMS_variables['Horizontal MSE Advection']
            + multi_experiment_GMS_variables['Vertical MSE Advection']
        ).sel(experiment=experiment, lat=slice(*latitude_bounds[experiment])).mean(dim=['lat', 'lon'])
        / (multi_experiment_GMS_variables['Vertical DSE Advection']).sel(experiment=experiment, lat=slice(*latitude_bounds[experiment])).mean(dim=['lat', 'lon'])
    for experiment in coords.experiments.values],
    dim=coords.experiments
)

intraseasonally_filtered_spatial_mean_GMS = {}
for experiment in coords.experiments.values:
    first_half = time_filter_data(spatial_mean_nGMS.sel(
        experiment=experiment,
        time=coords.first_half_subset_bounds
    ), slice(100, 20))
    second_half = time_filter_data(spatial_mean_nGMS.sel(
        experiment=experiment,
        time=coords.second_half_subset_bounds
    ), slice(100, 20))

    intraseasonally_filtered_spatial_mean_GMS[experiment] = xr.concat([first_half, second_half], dim='time').drop_sel(time=missing_days, errors='ignore')

multi_experiment_intraseasonally_filtered_spatial_mean_GMS = xr.concat(
    [intraseasonally_filtered_spatial_mean_GMS[experiment] for experiment in coords.experiments.values],
    dim=coords.experiments
)

# fontsize = 16
# plt.rcParams.update({'font.size':fontsize})
# fig = plt.figure(figsize=(9, 12))
# gs = fig.add_gridspec(3, 1, hspace=0.3)
# gs.update(left=0.05, right=0.95, bottom=0.05, top=0.95)

# axes_list = []
# axes_list.append(fig.add_subplot(gs[0]))
# axes_list.append(fig.add_subplot(gs[1]))
# axes_list.append(fig.add_subplot(gs[2]))

# for index, (ax, experiment) in enumerate(zip(axes_list, experiments_list)):
#     nGMS.sel(experiment=experiment).plot(levels=np.linspace(-10, 10, 21), extend='both', ax=ax)

# latitude_bounds = {}
# latitude_bounds['-4K'] = (-5, 5)
# latitude_bounds['0K'] = (-10, 10)
# latitude_bounds['4K'] = (-15, 15)

# reference_GMS = {}
# for experiment in experiments_list:
#     reference_GMS[experiment] = nGMS.sel(experiment=experiment, lat=slice(*latitude_bounds[experiment])).mean(dim=['lat', 'lon'])

# filtered_reference_GMS = {}
# for experiment in experiments_list:
#     first_half = time_filter_data(reference_GMS[experiment].sel(time=first_half_subset_bounds), slice(100, 20))
#     second_half = time_filter_data(reference_GMS[experiment].sel(time=second_half_subset_bounds), slice(100, 20))

#     filtered_reference_GMS[experiment] = xr.concat([first_half, second_half], dim='time')

# multi_experiment_filtered_reference_GMS = xr.concat(
#     [filtered_reference_GMS[experiment] for experiment in experiments_list],
#     dim=experiments_array
# )


In [ ]:
# savefig = False
# component_to_plot = 'Vertical'

# slope = {}
# intercept = {}
# import matplotlib.patheffects as pe

# fontsize = 16
# plt.rcParams.update({'font.size':fontsize})
# fig = plt.figure(figsize=(16, 6))
# gs = fig.add_gridspec(1, 2, wspace=0.2, width_ratios=[180, 90])
# gs.update(left=0.05, right=0.95, bottom=0.05, top=0.95)

# axes_list = []
# axes_list.append(fig.add_subplot(gs[0]))
# axes_list.append(fig.add_subplot(gs[1]))

# seed = 235913091
# random.seed(seed)
# random_sample = random.sample(range(0, len(denominator) + 1), 10000)

# numerator = {}
# denominator = {}



# for ax, data_type in zip(axes_list, (multi_experiment_GMS_variables, multi_experiment_intraseasonally_filtered_GMS_variables)):

#     for index, experiment in enumerate(experiments_list):

#         data_type['Total MSE Advection'] = data_type['Vertical MSE Advection'] + data_type['Horizontal MSE Advection']
#         denominator[experiment] = data_type['Vertical DSE Advection'].sel(experiment=experiment, lat=slice(*latitude_bounds[experiment]))
#         numerator[experiment] = (data_type[f"{component_to_plot} MSE Advection"]).sel(experiment=experiment, lat=slice(*latitude_bounds[experiment]))

#         denominator[experiment] = standardize_data(denominator[experiment].values.flatten(), dim='time', unit_variance=False)
#         print(experiment, np.max(denominator[experiment]))
#         numerator[experiment] = standardize_data(numerator[experiment].values.flatten(), dim='time', unit_variance=False)

#         slope[experiment], intercept[experiment] = np.polyfit(
#             y=numerator[experiment],
#             x=denominator[experiment],
#             deg=1
#         )

#         # ax.scatter(
#         #     denominator,
#         #     numerator,
#         #     c=bmh_colors(index+1),
#         #     edgecolor='k',
#         #     linewidths=0.2,
#         #     alpha=0.5,
#         #     label=f"{f'{experiments_array.sel(experiment=experiment)['name'].item()}':<4} (slope={slope[experiment]:>1.2f})"
#         # )
#         sns.kdeplot(
#             ax=ax,
#             x=denominator[experiment][random_sample],
#             y=numerator[experiment][random_sample],
#             fill=True,
#             color=bmh_colors(index+1),
#             alpha=0.75,
#         )

#         # ax.scatter(
#         #     denominator,
#         #     slope[experiment]*denominator+intercept[experiment],
#         #     marker='o',
#         #     color=bmh_colors(index+1),
#         #     edgecolor='k',
#         #     linewidths=0.5,
#         #     # mfc=bmh_colors(index+1),
#         #     # mec='k',
#         #     # mew=0.2,
#         #     # lw=4,
#         #     alpha=0.5,
#         #     label=f"{f'{experiments_array.sel(experiment=experiment)['name'].item()}':<4} (slope={slope[experiment]:>1.2f})",
#         #     zorder=10
#         # )
#         ax.plot(
#             denominator[experiment],
#             slope[experiment]*denominator[experiment]+intercept[experiment],
#             # marker='o',
#             color=bmh_colors(index+1),
#             # color='lightgray',
#             # edgecolor='k',
#             lw=3,
#             # mfc=bmh_colors(index+1),
#             # mec='k',
#             # mew=0.2,
#             # lw=4,
#             # alpha=0.5,
#             label=f"{f'{experiments_array.sel(experiment=experiment)['name'].item()}':<4} (slope={slope[experiment]:>1.2f})",
#             zorder=10,
#             # path_effects=[pe.Stroke(linewidth=4, foreground=bmh_colors(index+1)), pe.Normal()]
#         )

#     # ax.set_ylim(-20, 50)
#     # ax.set_aspect('equal')
#     ax.set_xlabel(r'Vertical DSE Advection (W m$^{-2}$)')
#     ax.set_ylabel(rf'{component_to_plot} MSE Advection (W m$^{-2}$)')
#     ax.legend(fontsize=12)
#     ax.axhline(y=0, color='gray', ls=':')
#     ax.axvline(x=0, color='gray', ls=':')

# axes_list[0].set_title(f'a) {component_to_plot} GMS Constituents', loc='left')
# axes_list[1].set_title(f'b) Intraseasonally-filtered \n {component_to_plot} GMS Constituents', loc='left')
# # axes_list[0].set_xlim(40, 300)
# # axes_list[0].set_ylim(-10, 40)
# # axes_list[0].set_xlim(-80, 100)
# # axes_list[0].set_ylim(-20, 20)
# # axes_list[1].set_xlim(-40, 50)
# # axes_list[1].set_ylim(-20, 20)
# # axes_list[1].set_aspect('equal')


# output_filename = f"Zero Centered {component_to_plot} GMS Constituents Scatter Plot"
# print(f"Output directory: {aquaplanet_output_directory}/gross_moist_stability_analysis/")
# print(f"Output filename: {output_filename}")
# if savefig:
#     print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
#     plt.savefig(
#         f"{aquaplanet_output_directory}/gross_moist_stability_analysis/{output_filename}.png",
#         dpi=500,
#         bbox_inches="tight",
#     )
#     print(rf"{'✔':>1}")
# else:
#     print("Not Saving")
#     plt.show()

In [ ]:
# component_to_plot = 'Vertical'

# # Mask and filter
# lat_mask = (
#     (multi_experiment_GMS_variables['Vertical MSE Advection'].lat >= latitude_bounds.sel(bound='lower')) &
#     (multi_experiment_GMS_variables['Vertical MSE Advection'].lat <= latitude_bounds.sel(bound='upper'))
# )

# denominator = multi_experiment_GMS_variables['Vertical DSE Advection'].where(lat_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
# denominator = standardize_data(denominator, dim='grid_points', unit_variance = False)

# numerator = multi_experiment_GMS_variables[f'{component_to_plot} MSE Advection'].where(lat_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
# numerator = standardize_data(numerator, dim='grid_points', unit_variance = False)

# slope = xr.zeros_like(experiments_array).astype('float64')
# intercept = xr.zeros_like(experiments_array).astype('float64')
# for experiment_index, experiment in enumerate(experiments_list):
#     slope[experiment_index], intercept[experiment_index] = np.polyfit(
#         y=numerator.sel(experiment=experiment).dropna(dim='grid_points'),
#         x=denominator.sel(experiment=experiment).dropna(dim='grid_points'),
#         deg=1
#     )

In [ ]:
savefig = False
recalculate = True

component_to_plot = 'Vertical'

all_variables_dict = {
    'Unfiltered': multi_experiment_GMS_variables,
    # 'Intraseasonally Filtered': multi_experiment_intraseasonally_filtered_GMS_variables,
    # 'MJO Filtered': multi_experiment_mjo_filtered_GMS_variables
}

if 'GMS_denominator' not in locals() or recalculate:
    GMS_denominator, GMS_numerator, GMS_estimate, GMS_intercept = {}, {}, {}, {}
    for dict_name, dict_values  in all_variables_dict.items():
        GMS_denominator[dict_name] = dict_values['Vertical DSE Advection'].where(coords.latitude_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
        GMS_denominator[dict_name] = standardize_data(GMS_denominator[dict_name], dim='grid_points', unit_variance = False)

        GMS_numerator[dict_name] = dict_values[f'{component_to_plot} MSE Advection'].where(coords.latitude_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
        GMS_numerator[dict_name] = standardize_data(GMS_numerator[dict_name], dim='grid_points', unit_variance = False)

        GMS_estimate[dict_name] = xr.zeros_like(coords.experiments).astype('float64')
        GMS_intercept[dict_name] = xr.zeros_like(coords.experiments).astype('float64')
        for experiment_index, experiment in enumerate(coords.experiments.values):
            GMS_estimate[dict_name][experiment_index], GMS_intercept[dict_name][experiment_index] = np.polyfit(
                y=GMS_numerator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
                x=GMS_denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
                deg=1
            )
            # num_aligned, denom_aligned = xr.align(GMS_numerator[dict_name].sel(experiment=experiment), GMS_denominator[dict_name].sel(experiment=experiment))
            # data = xr.concat([denom_aligned, num_aligned], dim="variable")
            # data = data.assign_coords(variable=["denom", "num"])
            # model = xe.single.EOF(n_modes=1)
            # model.fit(data, dim="grid_points")
            # eof = model.components()  # dims: (mode, variable)
            # direction = eof.sel(mode=1)
            # dx = direction.sel(variable="denom").item()
            # dy = direction.sel(variable="num").item()
            # GMS_estimate[dict_name][experiment_index] = dy / dx
            # # GMS_intercept
            # x_mean = denom_aligned.mean().item()
            # y_mean = num_aligned.mean().item()

            # GMS_intercept[dict_name][experiment_index] = y_mean - GMS_estimate[dict_name][experiment_index] * x_mean

fontsize = 8
plt.rcParams.update({'font.size':fontsize})
plt.rcParams['mathtext.fontset'] = 'dejavusans'
# fig = plt.figure(figsize=(5.5, 4.0))
fig = plt.figure(figsize=(5.5, 3.0))

gs = fig.add_gridspec(
    1, 2,
    wspace=0.1,
    width_ratios=[30, 1.5]
)
gs_plots = gs[0].subgridspec(1, 2, wspace=0.35)

axes_list = [
    fig.add_subplot(gs_plots[0]),
    fig.add_subplot(gs_plots[1]),
    # fig.add_subplot(gs_plots[2]),
]

gs_cbars = gs[1].subgridspec(1, 3, wspace=0.05)

cbar_axes = [
    fig.add_subplot(gs_cbars[0]),
    fig.add_subplot(gs_cbars[1]),
    fig.add_subplot(gs_cbars[2]),
]

for ax_index, (ax, (dict_name, dict_values))  in enumerate(zip(axes_list, all_variables_dict.items())):
    ax.set_title(f'{string.ascii_letters[ax_index]}) {dict_name}', loc='left', pad=5, fontsize=fontsize)
    for index, experiment in enumerate(coords.experiments.values):

        counts,x_edges,y_edges = np.histogram2d(
            x=GMS_denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
            y=GMS_numerator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
            bins=[100, 100],
        )
        im = ax.contourf(
            np.log10(counts.transpose()),
            # counts.transpose(),
            extent=[x_edges[0],x_edges[-1],y_edges[0],y_edges[-1]],
            cmap=config.EXPERIMENT_CMAPS[experiment],
            # levels = np.linspace(10, np.nanmax(counts), 21),
            levels=np.linspace(np.log10(10), 6, 21),
            alpha=0.9,
            zorder = 4-index
        )
        # cbar = fig.colorbar(im, cax=cbar_axis)
        if ax_index == 0:
            cbar = fig.colorbar(im, cax=cbar_axes[index])
            [cbar.ax.axhline(y=val, color='k', lw=1) for val in np.arange(1, 7, 1)]
            if experiment == '4K':
                cbar.set_ticks(np.arange(1, 7, 1))
                cbar.set_label(r"log$_{10}$(Count)", fontsize=fontsize)
            else:
                cbar.set_ticks([])

        ax.plot(
                GMS_denominator[dict_name].sel(experiment=experiment),
                (GMS_estimate[dict_name]*GMS_denominator[dict_name]+GMS_intercept[dict_name]).sel(experiment=experiment),
                color=bmh_colors(index+1),
                lw=3,
                # label=f"{experiments_array.sel(experiment=experiment)['name'].item():<4} (GMS={GMS_estimate[dict_name].sel(experiment=experiment):>1.2f})",
                label=f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]:<4} ({GMS_estimate[dict_name].sel(experiment=experiment):>1.2f})",
                zorder=10,
            )
        ax.axhline(y=0, color='gray', ls=':', zorder=10)
        ax.axvline(x=0, color='gray', ls=':', zorder=10)
        # ax.set_ylim(-500, 500)

    # ax.set_ylabel(rf'{component_to_plot} MSE Advection (W m$^{{{-2}}}$)', labelpad=5, fontsize=fontsize)
    ax.legend(loc='lower right', fontsize=fontsize-2)

# Unfiltered
axes_list[0].set_xlim(-1000, 4250)
axes_list[0].set_xticks(np.arange(-1000, 5000, 1000))
axes_list[0].set_ylim(-500, 500)

# # Intraseasonally Filtered
# axes_list[1].set_xlim(-1300, 1500)
# axes_list[1].set_xticks(np.arange(-1300, 1500+700, 700))
# axes_list[1].set_ylim(-215, 215)
# axes_list[1].set_yticks(np.arange(-200, 300, 100))

# MJO Filtered
axes_list[1].set_xlim(-700, 900)
axes_list[1].set_xticks(np.arange(-700, 900+400, 400))
axes_list[1].set_ylim(-105, 105)
axes_list[1].set_yticks(np.arange(-100, 125, 25))

for axis in axes_list:
    axis.set_aspect((axis.get_xlim()[1] - axis.get_xlim()[0])/(axis.get_ylim()[1] - axis.get_ylim()[0]))

axes_list[0].set_xlabel(r'Vertical DSE Advection (W m$^{-2}$)', labelpad=10, fontsize=fontsize)
axes_list[1].set_xlabel(r'Vertical DSE Advection (W m$^{-2}$)', labelpad=10, fontsize=fontsize)
axes_list[0].set_ylabel(r'Vertical MSE Advection (W m$^{-2}$)', labelpad=5, fontsize=fontsize)

# Save figure
output_filename = f"{component_to_plot} GMS Constituents Scatter Plot - Log(Histogram)"
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/")
print(f"Output filename: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")

plt.show()

In [ ]:
GMS_estimate

In [ ]:
savefig = False
recalculate = False
seed = 235913091
random.seed(seed)
sample_points = 10000

component_to_plot = 'Horizontal'

all_variables_dict = {
    'Unfiltered': multi_experiment_GMS_variables,
    'Intraseasonally Filtered': multi_experiment_intraseasonally_filtered_GMS_variables,
    'MJO Filtered': multi_experiment_mjo_filtered_GMS_variables
}


if 'denominator' not in locals() or recalculate:
    denominator, numerator, slope, intercept = {}, {}, {}, {}
    for dict_name, dict_values  in all_variables_dict.items():
        denominator[dict_name] = dict_values['Vertical DSE Advection'].where(coords.latitude_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
        denominator[dict_name] = standardize_data(denominator[dict_name], dim='grid_points', unit_variance = False)

        numerator[dict_name] = dict_values[f'{component_to_plot} MSE Advection'].where(coords.latitude_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
        numerator[dict_name] = standardize_data(numerator[dict_name], dim='grid_points', unit_variance = False)

        slope[dict_name] = xr.zeros_like(coords.experiments).astype('float64')
        intercept[dict_name] = xr.zeros_like(coords.experiments).astype('float64')
        for experiment_index, experiment in enumerate(coords.experiments.values):
            slope[dict_name][experiment_index], intercept[dict_name][experiment_index] = np.polyfit(
                y=numerator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
                x=denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
                deg=1
            )

fontsize = 16
plt.rcParams.update({'font.size':fontsize})
fig = plt.figure(figsize=(24, 18))
gs = fig.add_gridspec(3, 3, wspace=0.3, hspace=0.3, width_ratios=[1, 1, 1])
gs.update(left=0.05, right=0.95, bottom=0.05, top=0.95)

axes_list = []
axes_list.append(fig.add_subplot(gs[0,0]))
axes_list.append(fig.add_subplot(gs[0,1]))
axes_list.append(fig.add_subplot(gs[0,2]))
axes_list.append(fig.add_subplot(gs[1,0]))
axes_list.append(fig.add_subplot(gs[1,1]))
axes_list.append(fig.add_subplot(gs[1,2]))
axes_list.append(fig.add_subplot(gs[2,0]))
axes_list.append(fig.add_subplot(gs[2,1]))
axes_list.append(fig.add_subplot(gs[2,2]))

# for ax_index, (ax, (dict_name, dict_values))  in enumerate(zip(axes_list, all_variables_dict.items())):
for var_index, (dict_name, dict_values) in enumerate(all_variables_dict.items()):
    for experiment_index, experiment in enumerate(coords.experiments.values):
        # ax_index = 3*var_index + experiment_index
        ax_index = 3*experiment_index + var_index

        # random_sample = random.sample(range(0, len(denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points')) + 1), sample_points)
        # sample_percentage = len(random_sample)/len(denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'))

        # axes_list[ax_index].scatter(
        #     x=denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points')[::100],
        #     y=numerator[dict_name].sel(experiment=experiment).dropna(dim='grid_points')[::100],
        #     color=bmh_colors(experiment_index+1),
        #     alpha=0.5
        # )
        # sns.kdeplot(
        #     ax=ax,
        #     x=denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points')[random_sample],
        #     y=numerator[dict_name].sel(experiment=experiment).dropna(dim='grid_points')[random_sample],
        #     fill=True,
        #     color=bmh_colors(index+1),
        #     alpha=0.75
        # )
        counts,xbins,ybins = np.histogram2d(
            x=denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
            y=numerator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
            bins=[100, 100],
        )
        im = axes_list[ax_index].contourf(
            np.log10(counts.transpose()),
            # counts.transpose(),
            extent=[xbins[0],xbins[-1],ybins[0],ybins[-1]],
            cmap=config.EXPERIMENT_CMAPS[experiment],
            # levels = np.linspace(10, np.nanmax(counts), 21),
            levels=np.linspace(0.1, 6, 21),
            alpha=1.0
        )
        axes_list[ax_index].plot(
                denominator[dict_name].sel(experiment=experiment),
                (slope[dict_name]*denominator[dict_name]+intercept[dict_name]).sel(experiment=experiment),
                color=bmh_colors(experiment_index+1),
                lw=3,
                label=f"GMS={slope[dict_name].sel(experiment=experiment):>1.2f}",
                zorder=10,
            )

        if var_index == 0:
            axes_list[ax_index].set_xlim(-1500, 6000)
            # axes_list[ax_index].set_ylim(-500, 500)
            axes_list[ax_index].set_ylim(-1250, 1250)

        elif var_index == 1:
            axes_list[ax_index].set_xlim(-1500, 1500)
            # axes_list[ax_index].set_ylim(-250, 250)
            axes_list[ax_index].set_ylim(-400, 400)

        else:
            axes_list[ax_index].set_xlim(-1000, 1000)
            # axes_list[ax_index].set_ylim(-150, 150)
            axes_list[ax_index].set_ylim(-175, 175)

        axes_list[ax_index].set_xlabel(r'Vertical DSE Advection (W m$^{-2}$)')
        axes_list[ax_index].set_ylabel(rf'{component_to_plot} MSE Advection (W m$^{{{-2}}}$)')
        axes_list[ax_index].legend(fontsize=12, loc='upper left')
        axes_list[ax_index].axhline(y=0, color='gray', ls=':')
        axes_list[ax_index].axvline(x=0, color='gray', ls=':')
        axes_list[ax_index].set_title(
            (
                f"{string.ascii_letters[ax_index]}) "
                f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]} - "
                f"{dict_name}"
            ),
            loc='left'
        )

# axes_list[0].set_title(
#     (
#         f'a) {component_to_plot} GMS Constituents'
#         +(f", (Plot: {100*sample_percentage:0.1f}% Sample)" if sample_percentage < 1 else "")
#     ),
#     loc='left'
# )

output_filename = f"Zero Centered {component_to_plot} GMS Constituents Multi-Experiment Log(Histogram)"
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/")
print(f"Output filename: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight",
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

# Convective Adjustment Timescale

In [ ]:
savefig=False
# longitude_bounds = (179, 181)

# seed = 235913091
# random.seed(seed)
# random_sample = random.sample(range(0, len(flat_cwv) + 1), 1000)

convective_adjustment_timescale, slope, intercept = {}, {}, {}
[fig, ax] = plt.subplots(1, 1, figsize=(9,9))
for index, experiment in enumerate(coords.experiments.values):

    standardized_column_water_vapor = standardize_data(
            multi_experiment_variables_subset['Column Water Vapor'],
            dim='time',
            unit_variance=False
        ).sel(experiment=experiment, lat=slice(*coords.latitude_bounds[experiment]))

    standardized_precipitation = standardize_data(
            multi_experiment_variables_subset['Precipitation'],
            dim='time',
            unit_variance=False
        ).sel(experiment=experiment, lat=slice(*coords.latitude_bounds[experiment]))

    flat_cwv = standardized_column_water_vapor.values.flatten()
    flat_precip = standardized_precipitation.values.flatten()

    slope[experiment], intercept[experiment] = np.polyfit(
            y=flat_precip,
            x=flat_cwv,
            deg=1
        )
    convective_adjustment_timescale[experiment] = 24/slope[experiment]

    # Plot
    ax.set_title(
        # f"({tick_labeller([longitude_bounds[0]], 'lon')[0]}"
        # f"-{tick_labeller([longitude_bounds[1]], 'lon')[0]})"
        "ALL LON"
        # f"{tick_labeller([180], 'lon')[0]}"
    )
    # sns.kdeplot(
    #     ax=ax,
    #     x=flat_cwv[random_sample],
    #     y=flat_precip[random_sample],
    #     fill=True,
    #     color=bmh_colors(index+1),
    #     alpha=0.9
    # )
    # ax.scatter(
    #     x=flat_cwv[random_sample],
    #     y=flat_precip[random_sample],
    #     color=bmh_colors(index+1),
    #     alpha=0.5
    # )
    counts,xbins,ybins = np.histogram2d(
        x=flat_cwv,
        y=flat_precip,
        bins=[100, 100],
    )
    im = ax.contourf(
        np.log10(counts.transpose()),
        # counts.transpose(),
        extent=[xbins[0],xbins[-1],ybins[0],ybins[-1]],
        cmap=config.EXPERIMENT_CMAPS[experiment],
        # levels = np.linspace(10, np.nanmax(counts), 21),
        levels=np.linspace(0.1, 6, 21),
        alpha=1.0,
        zorder = 4-index
    )
    ax.plot(
            flat_cwv,
            slope[experiment]*flat_cwv+intercept[experiment],
            # marker='o',
            color=bmh_colors(index+1),
            # color='lightgray',
            # edgecolor='k',
            lw=3,
            # mfc=bmh_colors(index+1),
            # mec='k',
            # mew=0.2,
            # lw=4,
            # alpha=0.5,
            zorder=10,
            label=(
            rf"{f'{config.EXPERIMENT_DISPLAY_NAMES[experiment]}':<4}"
            rf" ($\tau_c$={convective_adjustment_timescale[experiment]:>1.1f} hr)"
            ),
    )

ax.axhline(y=0, color='gray', ls=':')
ax.axvline(x=0, color='gray', ls=':')

ax.set_ylabel(r'Precipitation (mm day$^{-1}$)')
ax.set_xlabel(r'Column Water Vapor (mm)')
# ax.set_xlim(-25, 25)
# ax.set_ylim(-25, 25)
# ax.set_xticks(np.arange(-25, 30, 5))
# ax.set_yticks(np.arange(-25, 30, 5))

ax.legend(loc='upper left')
ax.set_ylim(-50, 150)

output_filename = (
    "PRCP-CWV_ALL_LON_convective_adjustment_timescale"
)
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/")
print(f"Output file: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight"
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

In [ ]:
convective_adjustment_timescale

# Cloud-Radiative Feedback Parameter

In [ ]:
savefig = False

all_variables_dict = {
    'Unfiltered': multi_experiment_variables_subset,
    # 'Intraseasonally Filtered': multi_experiment_variables_filtered,
    # 'MJO Filtered': multi_experiment_variables_mjo_filtered,
}

N = np.size(multi_experiment_variables_subset['Precipitation'].sel(experiment='4K').values)
t_critical = t.ppf(0.95, df=N-2)

recalculate = True
if recalculate or 'r_estimate' not in locals():
    r_denominator, r_numerator, r_estimate, r_intercept = {}, {}, {}, {}
    for dict_name, dict_values  in all_variables_dict.items():
        # r_denominator[dict_name] = (config.HEAT_OF_VAPORIZATION/config.SECONDS_PER_DAY)*standardize_data(
        #     dict_values['Precipitation'],
        #     dim='time',
        #     unit_variance=False
        # ).where(lat_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
        r_denominator[dict_name] = (config.HEAT_OF_VAPORIZATION/config.SECONDS_PER_DAY)*dict_values['Precipitation'].where(coords.latitude_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
        r_denominator[dict_name] = standardize_data(
            r_denominator[dict_name],
            dim='grid_points',
            unit_variance = False
        )

        # r_numerator[dict_name] = standardize_data(
        #     # -(dict_values[f'Column Longwave Heating']+dict_values[f'Column Shortwave Heating']),
        #     dict_values[f'Outgoing Longwave Radiation'],
        #     dim='time', unit_variance=False
        # ).where(lat_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
        r_numerator[dict_name] = dict_values[f'Outgoing Longwave Radiation'].where(coords.latitude_mask, drop=True).stack(grid_points=(['time', 'lat', 'lon']))
        r_numerator[dict_name] = standardize_data(r_numerator[dict_name], dim='grid_points', unit_variance = False)

        # r_denominator[dict_name] = HEAT_OF_VAPORIZATION/SECONDS_PER_DAY*dict_values['Precipitation'].where(
        #     lat_mask, drop=True
        # ).stack(grid_points=(['time', 'lat', 'lon']))
        # r_numerator[dict_name] =  dict_values[f'Outgoing Longwave Radiation'].where(
        # lat_mask, drop=True
        # ).stack(grid_points=(['time', 'lat', 'lon']))

        r_estimate[dict_name] = xr.zeros_like(coords.experiments).astype('float64')
        r_intercept[dict_name] = xr.zeros_like(coords.experiments).astype('float64')
        for experiment_index, experiment in enumerate(coords.experiments.values):
            r_estimate[dict_name][experiment_index], r_intercept[dict_name][experiment_index] = np.polyfit(
                y=r_numerator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
                x=r_denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points'),
                deg=1
            )
            predicted_value = r_estimate[dict_name][experiment_index] * r_denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points')
            residual_sum_of_squares = ((r_estimate[dict_name][experiment_index] - predicted_value) ** 2).sum(dim=['grid_points'])
            residual_variance = residual_sum_of_squares / (N - 2)
            standard_error_slope = np.sqrt(residual_variance / (N - 1))
            t_statistic = r_estimate[dict_name][experiment_index] / standard_error_slope

fontsize = 14
plt.rcParams.update({'font.size':fontsize})
plt.rcParams['mathtext.fontset'] = 'dejavusans'
fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(1, 3, wspace=0.3)
gs.update(left=0.05, right=0.95, bottom=0.05, top=0.95)

axes_list = []
axes_list.append(fig.add_subplot(gs[0]))
axes_list.append(fig.add_subplot(gs[1]))
axes_list.append(fig.add_subplot(gs[2]))

experiment_mean_preciptiation = config.HEAT_OF_VAPORIZATION/config.SECONDS_PER_DAY*multi_experiment_variables_subset['Precipitation'].mean(dim=['time', 'lat', 'lon'])
experiment_mean_olr = multi_experiment_variables_subset['Outgoing Longwave Radiation'].mean(dim=['time', 'lat', 'lon'])

for ax_index, (ax, (dict_name, dict_values))  in enumerate(zip(axes_list, all_variables_dict.items())):
    ax.set_title(f'{string.ascii_letters[ax_index]}) {dict_name}', loc='left', pad=5, fontsize=fontsize+2)
    for index, experiment in enumerate(coords.experiments.values):

        counts,xbins,ybins = np.histogram2d(
            x=r_denominator[dict_name].sel(experiment=experiment).dropna(dim='grid_points')+experiment_mean_preciptiation.sel(experiment=experiment),
            y=r_numerator[dict_name].sel(experiment=experiment).dropna(dim='grid_points')+experiment_mean_olr.sel(experiment=experiment),
            bins=[100, 100],
        )
        im = ax.contourf(
            np.log10(counts.transpose()),
            # counts.transpose(),
            extent=[xbins[0],xbins[-1],ybins[0],ybins[-1]],
            cmap=config.EXPERIMENT_CMAPS[experiment],
            # levels = np.linspace(10, np.nanmax(counts), 21),
            levels=np.linspace(np.log10(10), 6, 21),
            alpha=0.9,
            zorder = 4-index
        )
        # ax.plot(
        #         r_denominator[dict_name].sel(experiment=experiment),
        #         (r_estimate[dict_name]*r_denominator[dict_name]+r_intercept[dict_name]).sel(experiment=experiment),
        #         color=bmh_colors(index+1),
        #         lw=3,
        #         label=f"{experiments_array.sel(experiment=experiment)['name'].item():<4} (r={-r_estimate[dict_name].sel(experiment=experiment):>1.2f})",
        #         zorder=10,
        #     )
        ax.legend(fontsize=10, loc='upper right', facecolor='white')
        # ax.axhline(y=0, color='gray', ls=':', zorder=10)
        # ax.axvline(x=0, color='gray', ls=':', zorder=10)

        ax.set_xlabel(r'L$_v$P (W m$^{-2}$)', labelpad=10, fontsize=fontsize)
        ax.set_ylabel(rf'OLR (W m$^{{{-2}}}$)', labelpad=5, fontsize=fontsize)
        # ax.set_ylabel(rf'$\langle$LW$\rangle$ (W m$^{{{-2}}}$)', labelpad=5)
        # ax.set_ylabel(rf'$\langle$LW$\rangle$ + $\langle$SW$\rangle$ (W m$^{{{-2}}}$)', labelpad=5)
        ax.set_facecolor('white')
        ax.grid(False)
        # ax.set_aspect('equal')

    # axes_list[0].set_ylim(-175, 125)
    # axes_list[0].set_xlim(-500, 2500)
    # axes_list[0].set_ylim(100, 400)
    # axes_list[0].set_xlim(-500, 2500)

    # axes_list[1].set_ylim(-125, 125)
    # axes_list[1].set_xlim(-1000, 1500)

    # axes_list[2].set_ylim(-75, 75)
    # axes_list[2].set_xlim(-600, 750)

for axis in axes_list:
    axis.set_aspect((axis.get_xlim()[1] - axis.get_xlim()[0])/(axis.get_ylim()[1] - axis.get_ylim()[0]))

# Save figure
output_filename = f"Cloud-Radiative Feedback Parameter Scatter Plot - Log(Histogram)"
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/")
print(f"Output filename: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/{output_filename}.png",
        dpi=500,
        bbox_inches="tight",
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

In [ ]:
r_estimate

In [ ]:
# timescale = 'Unfiltered'
timescale = 'Intraseasonally Filtered'
# timescale = 'MJO Filtered'
effective_GMS = {}

bar_width = 0.25

plt.style.use('bmh')
plt.rcParams.update({'font.size': 12, 'mathtext.fontset': 'dejavusans'})

# fig = plt.figure(figsize=(16, 12))
fig = plt.figure(figsize=(8.5, 11))
gs = GridSpec(3, 1, figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.25)

ax = []
ax.append(fig.add_subplot(gs[0]))
ax.append(fig.add_subplot(gs[1]))
ax.append(fig.add_subplot(gs[2]))

for filt_index, timescale in enumerate(['Unfiltered', 'Intraseasonally Filtered', 'MJO Filtered']):

    GMS_estimate[timescale].name = 'GMS'
    r_estimate[timescale].name = 'r'
    r_estimate[timescale] = np.abs(r_estimate[timescale])
    effective_GMS[timescale] = GMS_estimate[timescale]*(1+r_estimate[timescale]) - r_estimate[timescale]
    effective_GMS[timescale].name = 'Effective GMS'

    ax[filt_index].set_title(
        f"{string.ascii_letters[filt_index]}) {timescale}",
        loc='left',
        pad=10,
        fontsize=fontsize+2
    )

    processes_to_plot = [
        GMS_estimate[timescale],
        r_estimate[timescale],
        effective_GMS[timescale]
    ]

    x_positions = np.arange(len(processes_to_plot))
    for i, experiment in enumerate(coords.experiments.values):
        plot_label = (f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]}")
        ax[filt_index].bar(
            x_positions + i * bar_width,
            [process.sel(experiment=experiment).values for process in processes_to_plot],
            width=bar_width,
            label=plot_label,
            # color=[budget_term_attributes[process]['color'](0.3 + 0.35 * i) for process in processes_to_plot],
            edgecolor='#bcbcbc',
            lw=2,
        )

for axis in ax:
    axis.set_ylim(-0.075, 0.2125)
    axis.set_xticks(x_positions + bar_width, labels=[process.name for process in processes_to_plot], fontsize=fontsize)
    axis.axhline(y=0, color='#bcbcbc', lw=3)
    axis.grid(axis='x')
    [axis.axvline(x=(position + 3*bar_width), ls=':', color='#bcbcbc') for position in x_positions[:-1]]

    for spine in axis.spines.values():
        spine.set_edgecolor("#bcbcbc")
        spine.set_linewidth(3)

ax[0].legend(fontsize=fontsize)
plt.show()

In [ ]:
timescale = 'Unfiltered'
# timescale = 'Intraseasonally Filtered'
# timescale = 'MJO Filtered'
effective_GMS = {}

bar_width = 0.25

SAVE_FIG = True
PLOT_MODE = "slides"
FIG_LAYOUT = 'onecol'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.size'] = '16'
plt.rcParams['axes.labelsize'] = '16'
plt.rcParams['xtick.labelsize'] = '14'
plt.rcParams['ytick.labelsize'] = '14'
plt.rcParams['lines.linewidth'] = '2'

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))

gs = GridSpec(1, 1, figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.25)

ax = fig.add_subplot(gs[0])
# ax.append(fig.add_subplot(gs[1]))
# ax.append(fig.add_subplot(gs[2]))

GMS_estimate[timescale].name = 'GMS'
r_estimate[timescale].name = 'r'
r_estimate[timescale] = np.abs(r_estimate[timescale])
effective_GMS[timescale] = GMS_estimate[timescale]*(1+r_estimate[timescale]) - r_estimate[timescale]
effective_GMS[timescale].name = 'Effective GMS'

# ax.set_title(
#     f"{string.ascii_letters[filt_index]}) {timescale}",
#     loc='left',
#     pad=10,
#     fontsize=fontsize+2
# )

processes_to_plot = [
    GMS_estimate[timescale],
    r_estimate[timescale],
    effective_GMS[timescale]
]

x_positions = np.arange(len(processes_to_plot))
for i, experiment in enumerate(coords.experiments.values):
    plot_label = (f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]}")
    ax.bar(
        x_positions + i * bar_width,
        [process.sel(experiment=experiment).values for process in processes_to_plot],
        width=bar_width,
        label=plot_label,
        # color=[budget_term_attributes[process]['color'](0.3 + 0.35 * i) for process in processes_to_plot],
        edgecolor='#bcbcbc',
        lw=2,
    )

ax.set_ylim(-0.075, 0.2125)
ax.set_xticks(x_positions + bar_width, labels=[process.name for process in processes_to_plot], fontsize=fontsize)
ax.axhline(y=0, color='#bcbcbc', lw=3)
ax.grid(axis='x')
[ax.axvline(x=(position + 3*bar_width), ls=':', color='#bcbcbc') for position in x_positions[:-1]]

for spine in ax.spines.values():
    spine.set_edgecolor("#bcbcbc")
    spine.set_linewidth(3)

ax.legend(fontsize=fontsize)

output_filename = 'Effective GMS Bar Graph'
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/")
print(f"Output filename: {output_filename}")
if SAVE_FIG:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")

plt.show()

In [ ]:
GMS_estimate['Unfiltered']

In [ ]:
savefig = True

# r = np.linspace(0.05, 0.2, 100)
r = np.linspace(0.0, 0.2, 100)
# gamma = np.linspace(0.02, 0.2, 100)
gamma = np.linspace(0.0, 0.2, 100)

[R, G] = np.meshgrid(r, gamma)

G_eff = G*(1+R) - R

# r_estimate = {'Unfiltered':{0: -0.1579214379131003, 1: -0.13820106318009598, 2: -0.10081041163275163}}
r_estimate = {}
r_estimate['Unfiltered'] = xr.zeros_like(GMS_estimate['Unfiltered'])
r_estimate['Unfiltered'][0] = 0.1579214379131003
r_estimate['Unfiltered'][1] = 0.13820106318009598
r_estimate['Unfiltered'][2] = 0.10081041163275163

SAVE_FIG = True
PLOT_MODE = "publication"
FIG_LAYOUT = 'onecol'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.size'] = '12'
plt.rcParams['axes.labelsize'] = '12'
plt.rcParams['xtick.labelsize'] = '12'
plt.rcParams['ytick.labelsize'] = '12'
plt.rcParams['lines.linewidth'] = '2'

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))
gs = GridSpec(1, 1, figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.25)

ax = fig.add_subplot(gs[0])

cf = ax.contourf(
    R, G, G_eff,
    cmap='RdBu_r',
    norm=mcolors.CenteredNorm(vcenter=0),
    levels=np.arange(-0.2, 0.225, 0.025)
)

# Define colorbar width (in figure coordinates)
cbar_width = 0.5
pad = 1

cax = fig.add_axes(
    [
        ax.get_position().x1+0.05,
        ax.get_position().y0,
        0.04,
        ax.get_position().height
    ]
)

cbar = fig.colorbar(cf, cax=cax)
cbar.ax.hlines(0, 0, 1, colors='k', linewidth=1, linestyles='-')
cbar.set_label(r"GMS$_\text{eff}$")

cs = ax.contour(
    R,
    G,
    G_eff,
    colors='k',
    levels=[0],
    linewidths=1
)
ax.clabel(cs, colors='k', fmt='%1.0f', inline=True, manual=[(0.12, 0.1)])

ax.scatter(
    r_estimate['Unfiltered'],
    GMS_estimate['Unfiltered'],
    color=[bmh_colors(1), bmh_colors(2), bmh_colors(3)],
    s=350,
    marker='+',
    linewidths=2,
)
for index, experiment in enumerate(coords.experiments.values):
    ax.axvline(r_estimate['Unfiltered'].sel(experiment=experiment), color=bmh_colors(index+1), ls='-', alpha=0.2, linewidth=1)
    ax.axhline(GMS_estimate['Unfiltered'].sel(experiment=experiment), color=bmh_colors(index+1), ls='-', alpha=0.2, linewidth=1)

ax.set_xlabel('r')
ax.set_xticks(np.arange(0, 0.25, 0.05))
ax.set_xlim(0.0, 0.2)

ax.set_ylabel(r'GMS$_\text{v}$')
ax.set_yticks(np.arange(0, 0.25, 0.05))
ax.set_ylim(0.0, 0.2)
# ax.yaxis.set_major_locator(mticker.MaxNLocator(4, prune="lower"))
yticklabels = ax.get_yticklabels()
yticklabels[0].set_visible(False)

xticklabels = ax.get_xticklabels()
xticklabels[0].set_visible(False)

# Add a single shared "0" label at the origin
ax.text(
    0, 0, "0.00",
    ha="right", va="top",
    transform=ax.transData,
    fontsize=ax.xaxis.get_ticklabels()[0].get_fontsize()
)

ax.grid(False)
ax.set_aspect('equal', adjustable='box')

# Save figure
output_filename = f"Effective GMS Contourf Plot"
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/")
print(f"Output filename: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")
    plt.show()

In [ ]:
# data_directory = "/glade/derecho/scratch/sressel"
data_directory = r"/glade/campaign/univ/uwas0152/post_processed_data/"
# budget_to_load = 'MSE'
budget_to_load = 'moisture'
# budget_to_load = 'meridional_moisture_advection'
# data_source_to_load = '3-hourly_model-level'
# data_source_to_load = 'daily_pressure-level'
data_source_to_load = 'daily_model-level'

# processes_to_load = 'single decomposed advection term'
# processes_to_load = 'full budget with decomposed advection terms'
processes_to_load = 'full budget'

multi_experiment_budget_variables = {}
print("Loading budget variables...")
print(f"{'='*config.SEP_WIDTH}")
budget_data_location = f"{data_directory}/{budget_to_load}_budget_terms/{data_source_to_load}_data"
if budget_to_load == 'MSE':
    budget_variables_to_load = [
        'Moist Static Energy',
        'Moist Static Energy Tendency',
        'Zonal Advection',
        'Meridional Advection',
        'Vertical Advection',
        'Latent Heating',
        'Sensible Heating',
        'Longwave Heating',
        'Shortwave Heating',
        'Residual'
    ]

elif budget_to_load == 'moisture':
    if processes_to_load == 'full budget':
        budget_variables_to_load = [
            'Moisture',
            'Moisture Tendency',
            'Zonal Advection',
            'Meridional Advection',
            'Vertical Advection',
            'Evaporation',
            'Precipitation',
            'Residual'
        ]

    elif processes_to_load == 'single decomposed advection term':
        advection_type = 'Meridional Advection'
        budget_variables_to_load = [
            'Precipitation',
            'Moisture',
            'Moisture Tendency',
            f'{advection_type}',
            f'{advection_type} Mean wind mean moisture',
            f'{advection_type} Mean wind MJO moisture',
            f'{advection_type} Mean wind residial moisture',
            f'{advection_type} MJO wind mean moisture',
            f'{advection_type} MJO wind MJO moisture',
            f'{advection_type} MJO wind residual moisture',
            f'{advection_type} Residual wind mean moisture',
            f'{advection_type} Residual wind MJO moisture',
            f'{advection_type} Residual wind residual moisture',
        ]

    elif processes_to_load == 'full budget with decomposed advection terms':
        budget_variables_to_load = [
            'Moisture',
            'Moisture Tendency',
            'Evaporation',
            'Precipitation',
            f'Zonal Advection Mean wind MJO moisture',
            f'Zonal Advection Residual wind residual moisture',
            f'Meridional Advection MJO wind mean moisture',
            f'Meridional Advection Residual wind residual moisture',
            f'Vertical Advection MJO wind mean moisture',
            'Residual'
        ]

for index, variable_name in enumerate(budget_variables_to_load):
    data = xr.open_dataset(f"{budget_data_location}/multi_experiment_{variable_name.lower().replace(' ', '_')}.nc")
    for variable_data in data.data_vars.values():
        print(f"{f'({index+1}/{len(budget_variables_to_load)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")
        multi_experiment_budget_variables[variable_name] = variable_data
        print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
print(f"{'Subset budget variables':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

multi_experiment_budget_variables_subset = {}

for index, (variable_name, variable_data) in enumerate(multi_experiment_budget_variables.items()):
    print(f"{f'({index+1}/{len(multi_experiment_budget_variables)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

    multi_experiment_budget_variables_subset[variable_name] = variable_data.drop_sel(time=coords.missing_timesteps, errors='ignore')
    print(rf"{'✔':>1}")

# multi_experiment_budget_variables_subset[reference_variable] = variable_data.drop_sel(time=missing_timesteps, errors='ignore')

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
[fig, ax] = plt.subplots(1, 2, figsize=(16,6))
ax[0].scatter(
    GMS_estimate['Unfiltered'],
    config.SECONDS_PER_DAY*multi_experiment_budget_variables_subset['Vertical Advection'].sel(lat=slice(-15,15)).mean(dim=['time', 'lat', 'lon'])/multi_experiment_variables_subset['Precipitation'].sel(lat=slice(-15,15)).mean(dim=['time', 'lat', 'lon']),
    color=[bmh_colors(1), bmh_colors(2), bmh_colors(3)]
)

ax[1].scatter(
    -r_estimate['Unfiltered'],
    config.SECONDS_PER_DAY*multi_experiment_budget_variables_subset['Vertical Advection'].sel(lat=slice(-15,15)).mean(dim=['time', 'lat', 'lon'])/multi_experiment_variables_subset['Precipitation'].sel(lat=slice(-15,15)).mean(dim=['time', 'lat', 'lon']),
    color=[bmh_colors(1), bmh_colors(2), bmh_colors(3)]
)

## Combined GMS-r plot

In [ ]:
savefig = True
recalculate = False

component_to_plot = 'Vertical'

all_variables_dict = {
    'GMS': multi_experiment_GMS_variables,
    # 'r': multi_experiment_GMS_variables['Unfiltered']
    'r': multi_experiment_variables_subset
    # 'Intraseasonally Filtered': multi_experiment_intraseasonally_filtered_GMS_variables,
    # 'MJO Filtered': multi_experiment_mjo_filtered_GMS_variables
}

fontsize = 10
plt.rcParams.update({'font.size':fontsize})
plt.rcParams['mathtext.fontset'] = 'dejavusans'
# fig = plt.figure(figsize=(5.5, 4.0))
# fig = plt.figure(figsize=(5.5, 3.0))
fig = plt.figure(figsize=(16,6))

gs = fig.add_gridspec(
    1, 4,
    wspace=0.1,
    width_ratios=[30, 1.5, 30, 1]
)
gs_plots = gs[0].subgridspec(1, 2, wspace=0.45)

axes_list = [
    fig.add_subplot(gs_plots[0]),
    fig.add_subplot(gs_plots[1]),
    fig.add_subplot(gs[2]),
]

gs_cbars = gs[1].subgridspec(1, 3, wspace=0.05)

cbar_axes = [
    fig.add_subplot(gs_cbars[0]),
    fig.add_subplot(gs_cbars[1]),
    fig.add_subplot(gs_cbars[2]),
    fig.add_subplot(gs[3])
]


# axes_list[0].set_title(f'{string.ascii_letters[ax_index]}) {'GMS'}', loc='left', pad=5, fontsize=fontsize)
for index, experiment in enumerate(coords.experiments.values):

    GMS_counts,x_edges,y_edges = np.histogram2d(
        x=GMS_denominator['Unfiltered'].sel(experiment=experiment).dropna(dim='grid_points'),
        y=GMS_numerator['Unfiltered'].sel(experiment=experiment).dropna(dim='grid_points'),
        bins=[100, 100],
    )
    im = axes_list[0].contourf(
        np.log10(GMS_counts.transpose()),
        # counts.transpose(),
        extent=[x_edges[0],x_edges[-1],y_edges[0],y_edges[-1]],
        cmap=coords.EXPERIMENT_CMAPS[experiment],
        # levels = np.linspace(10, np.nanmax(counts), 21),
        levels=np.linspace(np.log10(10), 6, 21),
        alpha=0.9,
        zorder = 4-index
    )
    # cbar = fig.colorbar(im, cax=cbar_axis)
    cbar = fig.colorbar(im, cax=cbar_axes[index])
    [cbar.ax.axhline(y=val, color='k', lw=1) for val in np.arange(1, 7, 1)]
    if experiment == '4K':
        cbar.set_ticks(np.arange(1, 7, 1))
        cbar.set_label(r"log$_{10}$(Count)", fontsize=fontsize)
    else:
        cbar.set_ticks([])

    axes_list[0].plot(
        GMS_denominator['Unfiltered'].sel(experiment=experiment),
        (GMS_estimate['Unfiltered']*GMS_denominator['Unfiltered']+GMS_intercept['Unfiltered']).sel(experiment=experiment),
        color=bmh_colors(index+1),
        lw=3,
        label=f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]:<4} ({GMS_estimate['Unfiltered'].sel(experiment=experiment):>1.2f})",
        zorder=10,
    )

    r_counts,x_edges,y_edges = np.histogram2d(
        x=r_denominator['Unfiltered'].sel(experiment=experiment).dropna(dim='grid_points'),
        y=r_numerator['Unfiltered'].sel(experiment=experiment).dropna(dim='grid_points'),
        bins=[100, 100],
    )

    axes_list[1].contourf(
        np.log10(r_counts.transpose()),
        # counts.transpose(),
        extent=[x_edges[0],x_edges[-1],y_edges[0],y_edges[-1]],
        cmap=coords.EXPERIMENT_CMAPS[experiment],
        # levels = np.linspace(10, np.nanmax(counts), 21),
        levels=np.linspace(np.log10(10), 6, 21),
        alpha=0.9,
        zorder = 4-index
    )

    axes_list[1].plot(
        r_denominator['Unfiltered'].sel(experiment=experiment),
        (r_estimate['Unfiltered']*r_denominator['Unfiltered']+r_intercept['Unfiltered']).sel(experiment=experiment),
        color=bmh_colors(index+1),
        lw=3,
        label=f"{config.EXPERIMENT_DISPLAY_NAMES[experiment]:<4} ({-r_estimate['Unfiltered'].sel(experiment=experiment):>1.2f})",
        zorder=10,
    )

# Unfiltered
axes_list[0].set_xlim(-1000, 4250)
axes_list[0].set_xticks(np.arange(-1000, 5000, 1000))
axes_list[0].set_ylim(-500, 500)
axes_list[0].set_title(f'{string.ascii_letters[0]}) {'Vertical \nGross Moist Stability'}', loc='left', pad=5, fontsize=fontsize)
# axes_list[0].set_xlabel(r'Vertical DSE Advection (W m$^{-2}$)', labelpad=5, fontsize=fontsize)
# axes_list[0].set_ylabel(r'Vertical MSE Advection (W m$^{-2}$)', labelpad=3, fontsize=fontsize)
axes_list[0].set_xlabel(r'$\langle \omega \cdot \partial_p \,DSE \rangle$ (W m$^{-2}$)', labelpad=5, fontsize=fontsize)
axes_list[0].set_ylabel(r'$\langle \omega \cdot \partial_p \,MSE \rangle$ (W m$^{-2}$)', labelpad=5, fontsize=fontsize)
axes_list[0].legend(loc='lower right', fontsize=fontsize-2)

axes_list[1].set_ylim(-175, 125)
axes_list[1].set_xlim(-500, 2500)
axes_list[1].set_title(f'{string.ascii_letters[1]}) {'Cloud Radiative \nFeedback Parameter'}', loc='left', pad=5, fontsize=fontsize)
axes_list[1].set_xlabel(r'L$_v$P (W m$^{-2}$)', labelpad=5, fontsize=fontsize)
axes_list[1].set_ylabel(rf'OLR (W m$^{{{-2}}}$)', labelpad=3, fontsize=fontsize)
axes_list[1].legend(loc='upper right', fontsize=fontsize-2)

for ax in axes_list[:-1]:
    ax.axhline(y=0, color='gray', ls=':', zorder=10)
    ax.axvline(x=0, color='gray', ls=':', zorder=10)
    ax.set_aspect((ax.get_xlim()[1] - ax.get_xlim()[0])/(ax.get_ylim()[1] - ax.get_ylim()[0]))

axes_list[2].set_title(f'{string.ascii_letters[2]}) {'Effective Gross Moist Stability'}', loc='left', pad=5, fontsize=fontsize)
cf = axes_list[2].contourf(
    R, G, G_eff,
    cmap='RdBu_r',
    norm=mcolors.CenteredNorm(vcenter=0),
    levels=np.arange(-0.3, 0.325, 0.025)
)

# Define colorbar width (in figure coordinates)
cbar_width = 0.5
pad = 1


cbar = fig.colorbar(cf, cax=cbar_axes[-1])
cbar.ax.hlines(0, 0, 1, colors='k', linewidth=1, linestyles='-')
cbar.set_label(r"GMS$_\text{eff}$")

cs = axes_list[2].contour(
    R,
    G,
    G_eff,
    colors='k',
    levels=[0],
    linewidths=1
)
axes_list[2].clabel(cs, colors='k', fmt='%1.0f', inline=True, manual=[(0.12, 0.1)])

axes_list[2].scatter(
    -r_estimate['Unfiltered'],
    GMS_estimate['Unfiltered'],
    color=[bmh_colors(1), bmh_colors(2), bmh_colors(3)],
    s=500,
    marker='+',
    linewidths=3,
)
for index, experiment in enumerate(coords.experiments.values):
    axes_list[2].axvline(-r_estimate['Unfiltered'].sel(experiment=experiment), color=bmh_colors(index+1), ls='-', alpha=0.2, linewidth=1)
    axes_list[2].axhline(GMS_estimate['Unfiltered'].sel(experiment=experiment), color=bmh_colors(index+1), ls='-', alpha=0.2, linewidth=1)

axes_list[2].set_xlabel('r')
axes_list[2].set_xticks(np.arange(0, 0.25, 0.05))
axes_list[2].set_xlim(0.0, 0.2)

axes_list[2].set_ylabel(r'GMS$_\text{v}$')
axes_list[2].set_yticks(np.arange(0, 0.25, 0.05))
axes_list[2].set_ylim(0.0, 0.2)

axes_list[2].set_aspect('equal', adjustable='box')

# Save figure
output_filename = f"GMS, r, effective GMS plot - Log(Histogram)"
print(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/")
print(f"Output filename: {output_filename}")
if savefig:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/gross_moist_stability_analysis/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")

plt.show()